# 📦 Rapport — Suivi de prix e-commerce
**Site suivi :** [books.toscrape.com](https://books.toscrape.com)  
**Catégories :** Mystery · Science Fiction · Romance  

---

## Objectif
Ce notebook analyse les données collectées par `scripts/analyse.py` afin de :
- Visualiser les prix par catégorie
- Identifier les livres les moins / plus chers
- Suivre l'évolution des prix dans le temps
- Comparer la note des livres avec leur prix


In [ ]:
# ── Imports ──────────────────────────────────────────────────
import os, json, csv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from datetime import datetime

# Chemins relatifs au notebook (racine du projet)
CSV_FILE   = 'data/prix_historique.csv'
JSON_CACHE = 'data/derniere_capture.json'

print('Librairies chargées ✔')

## 1. Chargement des données
On lit le fichier CSV généré automatiquement par `analyse.py`.  
Chaque ligne correspond à une capture d'un livre à une date donnée.

In [ ]:
# Chargement du CSV historique
df = pd.read_csv(CSV_FILE, parse_dates=['date'])
df['prix'] = pd.to_numeric(df['prix'], errors='coerce')
df['etoiles'] = pd.to_numeric(df['etoiles'], errors='coerce')

print(f"Nombre total d'entrées : {len(df)}")
print(f"Période couverte       : {df['date'].min()} → {df['date'].max()}")
print(f"Catégories             : {df['categorie'].unique().tolist()}")
print()
df.head(10)

## 2. Statistiques descriptives
Quelques indicateurs clés sur les prix récoltés.

In [ ]:
# Dernière capture uniquement
derniere_date = df['date'].max()
df_last = df[df['date'] == derniere_date].copy()

print(f"=== Capture du {derniere_date} ===")
print(f"  Livres analysés   : {len(df_last)}")
print(f"  Prix moyen global : £{df_last['prix'].mean():.2f}")
print(f"  Prix médian       : £{df_last['prix'].median():.2f}")
print(f"  Prix minimum      : £{df_last['prix'].min():.2f}")
print(f"  Prix maximum      : £{df_last['prix'].max():.2f}")
print()

print("Prix moyen par catégorie :")
print(df_last.groupby('categorie')['prix'].mean().round(2).to_string())

## 3. Visualisation — Prix par catégorie
Le graphique ci-dessous compare le prix moyen de chaque catégorie lors de la dernière capture.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analyse des prix — dernière capture', fontsize=13, fontweight='bold')

# ── Barres : prix moyen par catégorie ────────────────────────
ax1 = axes[0]
moy_cat = df_last.groupby('categorie')['prix'].mean().sort_values()
couleurs = ['#4C72B0', '#DD8452', '#55A868']
bars = ax1.bar(moy_cat.index, moy_cat.values, color=couleurs[:len(moy_cat)], edgecolor='white')
ax1.bar_label(bars, fmt='£%.2f', padding=4, fontsize=10)
ax1.set_title('Prix moyen par catégorie')
ax1.set_ylabel('Prix (£)')
ax1.set_ylim(0, moy_cat.max() * 1.3)
ax1.tick_params(axis='x', rotation=10)

# ── Boxplot : distribution des prix ──────────────────────────
ax2 = axes[1]
categories = df_last['categorie'].unique()
data_box = [df_last[df_last['categorie'] == c]['prix'].values for c in categories]
bp = ax2.boxplot(data_box, labels=categories, patch_artist=True)
for patch, color in zip(bp['boxes'], couleurs):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_title('Distribution des prix par catégorie')
ax2.set_ylabel('Prix (£)')

plt.tight_layout()
plt.savefig('output/graphique_notebook.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Relation Prix ↔ Note (étoiles)
Y a-t-il une corrélation entre la note d'un livre et son prix ?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for cat, couleur in zip(df_last['categorie'].unique(), couleurs):
    sub = df_last[df_last['categorie'] == cat]
    ax.scatter(sub['etoiles'], sub['prix'], label=cat,
               color=couleur, s=90, alpha=0.8, edgecolors='white', linewidths=0.5)

ax.set_title('Prix vs Note des livres')
ax.set_xlabel('Note (★)')
ax.set_ylabel('Prix (£)')
ax.set_xticks([1, 2, 3, 4, 5])
ax.legend()
plt.tight_layout()
plt.show()

# Corrélation
corr = df_last['etoiles'].corr(df_last['prix'])
print(f"Corrélation étoiles / prix : {corr:.3f}")
if abs(corr) < 0.3:
    print("→ Faible corrélation : la note n'explique pas le prix.")
elif abs(corr) < 0.6:
    print("→ Corrélation modérée.")
else:
    print("→ Forte corrélation !")

## 5. Évolution des prix dans le temps
Si le script a été exécuté plusieurs fois, cette section montre l'évolution du prix moyen par catégorie.

In [ ]:
# Prix moyen par date et catégorie
evol = df.groupby(['date', 'categorie'])['prix'].mean().reset_index()

nb_dates = evol['date'].nunique()
if nb_dates < 2:
    print("ℹ️  Seulement une capture disponible.")
    print("   Lance analyse.sh plusieurs fois (ex. : 2-3 jours de suite)")
    print("   pour voir l'évolution des prix.")
else:
    fig, ax = plt.subplots(figsize=(11, 5))
    for cat, couleur in zip(evol['categorie'].unique(), couleurs):
        sub = evol[evol['categorie'] == cat]
        ax.plot(sub['date'], sub['prix'], marker='o', label=cat, color=couleur, linewidth=2)
    ax.set_title('Évolution du prix moyen par catégorie')
    ax.set_xlabel('Date')
    ax.set_ylabel('Prix moyen (£)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %H:%M'))
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

## 6. Top 5 livres les moins chers
Classement des livres les moins chers de la dernière capture.

In [ ]:
top5 = df_last.nsmallest(5, 'prix')[['titre', 'categorie', 'prix', 'etoiles', 'disponible']]
top5.columns = ['Titre', 'Catégorie', 'Prix (£)', 'Note (★)', 'Disponibilité']
top5.reset_index(drop=True, inplace=True)
top5.index += 1
print("=== Top 5 livres les moins chers ===")
print(top5.to_string())

## 7. Conclusion

> **Complète cette section avec tes observations personnelles.**

Quelques pistes de réflexion :
- Quelle catégorie est la plus chère / la moins chère ?
- Y a-t-il une corrélation entre la note et le prix ?
- Quels livres ont vu leur prix changer entre deux captures ?
- Comment améliorer ce projet (ex. : ajouter plus de catégories, planifier avec `cron`) ?

**Planification automatique avec `cron`** (optionnel) :
```bash
# Exécuter le script chaque jour à 8h00
crontab -e
0 8 * * * /bin/bash /chemin/vers/scripts/analyse.sh >> /tmp/suivi_prix.log 2>&1
```
